# Selección de variables: experimentos seleccionados

Bitácora UL05 asignada a Keber. Los resultados de filtros de la actividad 1 tienen fuga de información; la comparación wrapper separa entrenamiento y prueba antes de seleccionar. Extracto histórico sin salidas ni datos.


## Multicolinealidad


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calcular la matriz de correlación
corr_matrix = X_data.corr()

# Graficar el heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlación de las Características")
plt.show()

In [ ]:
for i, feature in enumerate(feature_names):
    max_corr = max(abs(corr_matrix[feature][corr_matrix[feature] < 1]))  # Evitar la correlación de 1 consigo misma
    print(f"{feature:20s} | F-value: {f_value[0][i]:7.2f} | Máxima correlación con otra variable: {max_corr:.2f}")

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Crear un DataFrame para calcular VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X_data.columns
vif_data["VIF"] = [variance_inflation_factor(X_data.values, i) for i in range(X_data.shape[1])]

# Mostrar resultados
print(vif_data)

In [ ]:
def train_and_evaluate(X, y, description):
    # Dividir en conjunto de entrenamiento y prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

    # Instanciar y entrenar el modelo
    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)

    # Hacer predicciones
    y_pred = model.predict(X_test)

    # Calcular precisión
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Precisión con {description}: {accuracy:.4f}')

    return accuracy

# Modelo con todas las características
acc_full = train_and_evaluate(X_data, y_data, "todas las características")

# Modelo sin Sepal length
X_no_sepal_length = X_data.drop(columns=["sepal length (cm)"])
acc_no_sepal_length = train_and_evaluate(X_no_sepal_length, y_data, "sin sepal length")

# Modelo sin Petal length
X_no_petal_length = X_data.drop(columns=["petal length (cm)"])
acc_no_petal_length = train_and_evaluate(X_no_petal_length, y_data, "sin petal length")

# Modelo sin Sepal length ni Petal length
X_no_both = X_data.drop(columns=["sepal length (cm)", "petal length (cm)"])
acc_no_both = train_and_evaluate(X_no_both, y_data, "sin sepal length ni petal length")

In [ ]:
X_sepal_width = X_data[["sepal width (cm)"]]
acc_only_sepal_width = train_and_evaluate(X_sepal_width, y_data, "solo Sepal width")

X_petal_width = X_data[["petal width (cm)"]]
acc_only_sepal_width = train_and_evaluate(X_petal_width, y_data, "solo Petal width")


## Comparación de métodos wrapper


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

def evaluate_wrapper_method(X, y, method_name, model_type="logistic"):
    """
    Aplica un método wrapper de selección de características y evalúa su precisión
    con Regresión Logística o Naive Bayes.

    model_type: "logistic" para Regresión Logística, "naive_bayes" para GaussianNB
    """
    # Dividir en entrenamiento y prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

    # Definir el modelo
    if model_type == "logistic":
        model = LogisticRegression(max_iter=5000)
    elif model_type == "naive_bayes":
        model = GaussianNB()
    else:
        raise ValueError("model_type debe ser 'logistic' o 'naive_bayes'")

    # Selección de características según el método
    if method_name == "EFS":
        selector = EFS(estimator=model, min_features=1, max_features=4, scoring="accuracy", cv=5)
    elif method_name == "SFS":
        selector = SFS(estimator=model, k_features=(1, 4), forward=True, scoring="accuracy", cv=5)
    elif method_name == "SBS":
        selector = SFS(estimator=model, k_features=(1, 4), forward=False, scoring="accuracy", cv=5)
    else:
        raise ValueError("method_name debe ser 'EFS', 'SFS' o 'SBS'")

    # Entrenar el selector de características
    selector.fit(X_train, y_train)

    # Obtener subconjunto de características seleccionadas
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)

    # Entrenar el modelo con las características seleccionadas
    model.fit(X_train_selected, y_train)

    # Hacer predicciones
    y_pred = model.predict(X_test_selected)

    # Calcular precisión
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Precisión con {method_name} y {model_type}: {accuracy:.4f}')

    # Obtener las características seleccionadas correctamente según el método
    if method_name == "EFS":
        selected_features = selector.best_feature_names_
    else:  # Para SFS y SBS
        selected_features = selector.k_feature_names_

    print(f'Mejores características seleccionadas ({method_name}): {selected_features}')

    return accuracy, selected_features


In [ ]:
# Cargar dataset
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
X_data_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_data_bc = data.target

# Evaluar cada método con ambos modelos
methods = ["EFS", "SFS", "SBS"]
models = ["logistic", "naive_bayes"]

results = {}

for method in methods:
    for model in models:
        acc, selected_features = evaluate_wrapper_method(X_data_bc, y_data_bc, method, model)
        results[(method, model)] = (acc, selected_features)


In [ ]:
# Mostrar resumen de resultados
print("")
print("Comparación de Resultados:")
for key, value in results.items():
    method, model = key
    acc, features = value
    print(f"Método: {method} | Modelo: {model} | Precisión: {acc:.4f} | Features: {features}")


## SelectFromModel con tres estimadores


In [ ]:
# Tu código aquí ...
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_data_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_data_bc = data.target

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_data_bc, y_data_bc, test_size=0.3, random_state=0)

# Revisar los datos
X_train.shape, X_test.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

# Instanciar el modelo Random Forest
rfc = RandomForestClassifier(random_state=123, criterion='gini')

# Instanciar SelectFromModel
sfm = SelectFromModel(estimator=rfc)

# Entrenar el selector
sfm.fit(X_train, y_train)

# Transformar los datos para quedarnos solo con las características más importantes
X_important_train = sfm.transform(X_train)
X_important_test = sfm.transform(X_test)

# Ver cuántas características fueron seleccionadas
print(f'Número de características antes de la transformación: {X_train.shape[1]}')
print(f'Número de características después de la transformación: {X_important_train.shape[1]}')

# Entrenar el modelo con todas las características
rfc_full = RandomForestClassifier(random_state=0, criterion='gini')
rfc_full.fit(X_train, y_train)
pred_full = rfc_full.predict(X_test)
acc_full = accuracy_score(y_test, pred_full)

# Entrenar el modelo con características seleccionadas
rfc_lim = RandomForestClassifier(random_state=0, criterion='gini')
rfc_lim.fit(X_important_train, y_train)
pred_lim = rfc_lim.predict(X_important_test)
acc_lim = accuracy_score(y_test, pred_lim)

# Comparar precisión
print(f'Precisión con todas las características: {acc_full:.4f}')
print(f'Precisión con características seleccionadas: {acc_lim:.4f}')


In [ ]:
from sklearn.svm import SVC

# Instanciar el modelo SVM
svm = SVC(kernel='linear', random_state=123)

# Instanciar SelectFromModel
sfm_svm = SelectFromModel(estimator=svm)

# Entrenar el selector
sfm_svm.fit(X_train, y_train)

# Transformar los datos para quedarnos solo con las características más importantes
X_important_train_svm = sfm_svm.transform(X_train)
X_important_test_svm = sfm_svm.transform(X_test)

# Ver cuántas características fueron seleccionadas
print(f'Número de características antes de la transformación: {X_train.shape[1]}')
print(f'Número de características después de la transformación: {X_important_train_svm.shape[1]}')

# Entrenar el modelo con todas las características
svm_full = SVC(kernel='linear', random_state=123)
svm_full.fit(X_train, y_train)
pred_full_svm = svm_full.predict(X_test)
acc_full_svm = accuracy_score(y_test, pred_full_svm)

# Entrenar el modelo con características seleccionadas
svm_lim = SVC(kernel='linear', random_state=123)
svm_lim.fit(X_important_train_svm, y_train)
pred_lim_svm = svm_lim.predict(X_important_test_svm)
acc_lim_svm = accuracy_score(y_test, pred_lim_svm)

# Comparar precisión
print(f'Precisión con todas las características (SVM): {acc_full_svm:.4f}')
print(f'Precisión con características seleccionadas (SVM): {acc_lim_svm:.4f}')


In [ ]:
from sklearn.linear_model import LogisticRegression

# Instanciar el modelo LASSO (Regresión logística con penalización L1)
lasso = LogisticRegression(penalty='l1', solver='liblinear', random_state=123, max_iter=5000) # Se añade parámetro max_iter por ConvergenceWarning

# Instanciar SelectFromModel
sfm_lasso = SelectFromModel(estimator=lasso)

# Entrenar el selector
sfm_lasso.fit(X_train, y_train)

# Transformar los datos para quedarnos solo con las características más importantes
X_important_train_lasso = sfm_lasso.transform(X_train)
X_important_test_lasso = sfm_lasso.transform(X_test)

# Ver cuántas características fueron seleccionadas
print(f'Número de características antes de la transformación: {X_train.shape[1]}')
print(f'Número de características después de la transformación: {X_important_train_lasso.shape[1]}')

# Entrenar el modelo con todas las características
lasso_full = LogisticRegression(penalty='l1', solver='liblinear', random_state=123, max_iter=5000) # Se añade parámetro max_iter por ConvergenceWarning
lasso_full.fit(X_train, y_train)
pred_full_lasso = lasso_full.predict(X_test)
acc_full_lasso = accuracy_score(y_test, pred_full_lasso)

# Entrenar el modelo con características seleccionadas
lasso_lim = LogisticRegression(penalty='l1', solver='liblinear', random_state=123, max_iter=5000) # Se añade parámetro max_iter por ConvergenceWarning
lasso_lim.fit(X_important_train_lasso, y_train)
pred_lim_lasso = lasso_lim.predict(X_important_test_lasso)
acc_lim_lasso = accuracy_score(y_test, pred_lim_lasso)

# Comparar precisión
print(f'Precisión con todas las características (LASSO): {acc_full_lasso:.4f}')
print(f'Precisión con características seleccionadas (LASSO): {acc_lim_lasso:.4f}')
